### 1. Library Imports
Importing all required PyTorch, TorchVision, and utility modules for training and preprocessing.

In [24]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import torch.ao.quantization as quantization

print("Libraries imported successfully. PyTorch version:", torch.__version__)

Libraries imported successfully. PyTorch version: 2.11.0+cpu


### 2. Dataset Setup & Class Mapping
Defining the dataset paths and setting up the class indices for 3-class classification: Healthy Potato, Diseased Potato, and Not a Potato Leaf.

In [13]:
# Set your dataset directory path
DATASET_DIR = "./dataset"

# Target classes
CLASSES = ["healthy_potato", "diseased_potato", "non_potato"]

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### 3. Transfer Learning Model Architecture (MobileNetV2)
Loading a lightweight pre-trained MobileNetV2 network, freezing its feature extractor, and replacing its classifier head for your 3 target classes.

In [18]:
import torchvision.models as models

# 1. Load a lightweight pre-trained model (MobileNetV2)
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# 2. Freeze the feature extractor layers so we only train the final classifier
for param in model.parameters():
    param.requires_grad = False

# 3. Replace the final classifier head with our 3-class output
num_detected_classes = len(full_dataset.classes)  # Should be 3
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(model.last_channel, num_detected_classes)
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()

# Only optimize the parameters of the new classification head
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print(f"Transfer learning model initialized for classes: {full_dataset.classes}")

Transfer learning model initialized for classes: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


### 4. Model Training & Validation Loop
Running the training loop over multiple epochs with MobileNetV2, tracking losses, and evaluating validation accuracy.

In [19]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = (correct_train / total_train) * 100

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = (correct_val / total_val) * 100

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | "
        f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}%"
    )

Epoch [1/10] Train Loss: 0.5626 | Train Acc: 83.03% | Val Loss: 0.3321 | Val Acc: 93.04%
Epoch [2/10] Train Loss: 0.2636 | Train Acc: 93.67% | Val Loss: 0.2270 | Val Acc: 95.13%
Epoch [3/10] Train Loss: 0.2212 | Train Acc: 94.07% | Val Loss: 0.1853 | Val Acc: 96.06%
Epoch [4/10] Train Loss: 0.1860 | Train Acc: 94.60% | Val Loss: 0.1680 | Val Acc: 94.66%
Epoch [5/10] Train Loss: 0.1828 | Train Acc: 94.07% | Val Loss: 0.1466 | Val Acc: 95.59%
Epoch [6/10] Train Loss: 0.1637 | Train Acc: 95.18% | Val Loss: 0.1494 | Val Acc: 95.59%
Epoch [7/10] Train Loss: 0.1446 | Train Acc: 95.53% | Val Loss: 0.1267 | Val Acc: 95.82%
Epoch [8/10] Train Loss: 0.1384 | Train Acc: 95.64% | Val Loss: 0.1328 | Val Acc: 96.06%
Epoch [9/10] Train Loss: 0.1201 | Train Acc: 96.63% | Val Loss: 0.1167 | Val Acc: 95.59%
Epoch [10/10] Train Loss: 0.1202 | Train Acc: 96.75% | Val Loss: 0.1242 | Val Acc: 96.06%


### 5. Weights Export
Saving the trained model state dictionary to disk for stream inference and downstream quantization steps.

In [22]:
# Your exact absolute weights directory path
SAVE_DIR = r"C:\Users\User\OneDrive\Desktop\edge-ai-coprocessor\models\weights"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save with a clear 3-class filename
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "potato_model_3class.pth")

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"New 3-class model successfully saved at: {MODEL_SAVE_PATH}")

New 3-class model successfully saved at: C:\Users\User\OneDrive\Desktop\edge-ai-coprocessor\models\weights\potato_model_3class.pth


### 6. INT8 Post-Training Quantization
Converting your floating-point MobileNetV2 model weights to INT8 to reduce memory footprint and match the hardware constraints of the Altera FPGA coprocessor.

In [25]:
# Ensure model is in evaluation mode for quantization
model.eval()

# Specify quantization engine (fbgemm for x86 CPUs or qnnpack)
backend = "qnnpack"
model.qconfig = quantization.get_default_qconfig(backend)

# Prepare the model for quantization (inserts observers)
prepared_model = quantization.prepare(model, inplace=False)

# Optional: Run a quick calibration pass with a few validation batches to calibrate scales/zero-points
print("Calibrating model for INT8 quantization...")
with torch.no_grad():
    for i, (inputs, _) in enumerate(val_loader):
        if i > 5:  # Just a few batches for quick calibration
            break
        prepared_model(inputs)

# Convert to final quantized model
quantized_model = quantization.convert(prepared_model, inplace=False)
print("Model successfully quantized to INT8!")

# Save the quantized model weights
QUANT_SAVE_PATH = r"C:\Users\User\OneDrive\Desktop\edge-ai-coprocessor\models\export\quantized_weights.pth"
os.makedirs(os.path.dirname(QUANT_SAVE_PATH), exist_ok=True)
torch.save(quantized_model.state_dict(), QUANT_SAVE_PATH)
print(f"Quantized weights saved to: {QUANT_SAVE_PATH}")

C:\Users\User\AppData\Local\Temp\ipykernel_18844\641157409.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model = quantization.prepare(model, inplace=False)


Calibrating model for INT8 quantization...


C:\Users\User\AppData\Local\Temp\ipykernel_18844\641157409.py:20: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantization.convert(prepared_model, inplace=False)


Model successfully quantized to INT8!
Quantized weights saved to: C:\Users\User\OneDrive\Desktop\edge-ai-coprocessor\models\export\quantized_weights.pth


### 7: Final Evaluation & Classification Report

In [26]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

print("Running final evaluation on validation set...")
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Map indices back to class names
class_names = full_dataset.classes
print("\n--- Classification Report ---")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(all_labels, all_preds))

Running final evaluation on validation set...

--- Classification Report ---
                       precision    recall  f1-score   support

Potato___Early_blight       0.98      0.98      0.98       194
 Potato___Late_blight       0.97      0.94      0.96       206
     Potato___healthy       0.77      0.97      0.86        31

             accuracy                           0.96       431
            macro avg       0.91      0.96      0.93       431
         weighted avg       0.96      0.96      0.96       431


--- Confusion Matrix ---
[[190   4   0]
 [  3 194   9]
 [  0   1  30]]
